## This Notebook will be used to create a more sophisticated elo than model 1, surface specific elos, weighted elo based
## on tournament category and dynmaic elo based on amount of games played by the player

In [ ]:
from pathlib import Path
import pandas as pd 
import numpy as np 

## without numpy we end up doing way too slow on operations in python

In [7]:
INTERIM_DIR = Path("../data/interim")

df = pd.read_csv(INTERIM_DIR / "atp_all_matches_raw.csv", low_memory = False)

## le colonne da tenere sono date category name winner loser bestof match_num
KEEP_COLS = [
    "tourney_date",
    "tourney_id",
    "tourney_level",
    "match_num",
    "winner_id",
    "loser_id",
    "best_of"
]

df = df[KEEP_COLS].copy()
## copy() crea una copia indipendente del DataFrame
## se non faccimamo cosi andiamo a modificare il DataFrame originale
## good practice usarlo dopo i filtraggi come sopra

In [8]:
print(df.shape)
df.head()

(941649, 7)


,tourney_date,tourney_id,tourney_level,match_num,winner_id,loser_id,best_of
0,18770709,1877-540,G,1,113987,114009,5
1,18770709,1877-540,G,2,114146,113977,5
2,18770709,1877-540,G,3,114147,114148,5
3,18770709,1877-540,G,4,114149,114150,5
4,18770709,1877-540,G,5,113953,114151,5


In [9]:
## qui definiamo i tourney weights, cosi da avere un elo piu informato

TOURNEY_WEIGHTS = {
    "G" : 1.5, #GS
    "M" : 1.2, #M1000,
    "A" : 1.0, #250/500
    "C" : 0.8, #CHALL
    "F" : 0.5, #FUTURES
}

INITIAL_RATING = 1500.0

def get_k(n_matches):
    return 250/ ((n_matches + 5) **0.4)

def expected_score(r_a, r_b):
    return 1/ (1 + 10** ((r_b - r_a) / 400))

In [ ]:
ratings = {}
n_matches = {}

winner_elo_pre = []
loser_elo_pre = []

for _, row in df.iterrows():
    w_id = row["winner_id"]
    l_id = row["loser_id"]

    r_w = ratings.get(w_id, INITIAL_RATING)
    r_l = ratings.get(l_id, INITIAL_RATING)
    

    winner_elo_pre.append(r_w)
    loser_elo_pre.append(r_l)

    e_w = expected_score(r_w, r_l)
    e_l = 1 - e_w

    n_w = n_matches.get(w_id, 0)
    n_l = n_matches.get(l_id, 0)

    weight = TOURNEY_WEIGHTS.get(row["tourney_level"], 1.0)
    k_w = get_k(n_w) * weight
    k_l = get_k(n_l) * weight

    ratings[w_id] = r_w + k_w * (1- e_w)
    ratings[l_id] = r_l + k_l * (0 - e_l)

    n_matches[w_id] = n_w + 1
    n_matches[l_id] = n_l + 1

df["winner_elo"] = winner_elo_pre
df["loser_elo"] = loser_elo_pre